### 유전 알고리즘

- 유전 알고리즘 : 생물들의 진화 현상을 모방하여 최적화 문제를 풀기 위한 알고리즘
	- 기존의 신약 후보 중 우월한 부모 세대 선정
	- 우월한 부모 세대 2개를 합치기 & 부모 세대 수정으로 자식 세대 구성
	- 최종적으로 가장 우월한 자식 선정

In [17]:
import pandas as pd 
import random
import selfies as sf

df = pd.read_csv('항암제 모든 예측 포함.csv')

In [ ]:
from minimax_backend.prediction_all import toxic_predict, lipinski_rule, predict_pKi, predict_pKd
from rdkit import Chem
smiles = df['dnew_smiles'][0]
mol = Chem.MolFromSmiles(smiles)

def predict_feature(smiles, mol):
	toxic_score = toxic_predict(smiles) # 이전보다 낮아야 함
	lipinski_value = lipinski_rule(mol) # molecule_weight는 500 이하, logp는 5이하, qed는 높아야 함, hbd 5개 이하, hba 10개 이하
	pki_value = predict_pKi(smiles)[1] # 이전보다 높아야 함 
	pkd_value = predict_pKd(smiles)[1] # 이전보다 높아야 함 
	key = ['독성 점수', '분자량', 'logp', 'qed', 'hbd', 'hba','pki','pkd']
	value = [toxic_score] + lipinski_value + [pki_value] + [pkd_value]
	value = [round(i,2) for i in value]
	return dict(zip(key,value))

In [19]:
import json

# 파일 열기
with open("D:\minimax원본\molecule_optimization\\for_predict_file\\vocab.json", "r", encoding="utf-8") as f:
    data = json.load(f)   # JSON → 파이썬 딕셔너리/리스트 변환

In [20]:
target = smiles
gen_size = 5

# randch: 무작위 문자 하나 생성
def randch():
    rand_mole = random.sample([i for i in data.keys() if i not in ['[SOS]','[EOS]','[PAD]']],1)
    return rand_mole[0]

# make_random_chrono: 무작위 해 하나 생성
def make_random_chrono(target): # 분자의 일부를 삭제하거나 더하는 걸 랜덤으로 수행함
	chrono = list(sf.split_selfies(sf.encoder(target)))
	for i in range(3):
		x = random.randint(0,1)
		if x == 0:
			remove_molecule = random.randint(0, len(chrono)-1)
			chrono.remove(chrono[remove_molecule])
		else:
			chrono.insert(random.randint(0,len(chrono)),randch())

	return sf.decoder(''.join(chrono))

# make_random_generation: 초기 해집단 생성(= 해를 gen_size개만큼 생성) = 5개
def make_random_generation(target):
    return [make_random_chrono(target) for _ in range(gen_size)]

In [21]:
# get_fitness: 적합도 계산 
# 분자 smiles에서는 기존 독성 예측 등의 값과 해집단의 예측값을 비교해서 차이가 많이 나면 좋은거로
def get_fitness(smiles, mol, orig_features):
	features = predict_feature(smiles, mol)          

	# Lipinski 기준 만족 여부 (True=1, False=0)
	lip_pass = (features['분자량'] <= 500 and
				features['logp'] <= 5 and
				features['hbd'] <= 5 and
				features['hba'] <= 10)

	# 간단한 스코어링: 낮은 tox → 1/tox, 높은 qed, 높은 pKi, 높은 pKd
	score = sum([-(features['독성 점수'] - orig_features['독성 점수']),
          	(features['qed'] - orig_features['qed']),
            (features['pki'] - orig_features['pki']),
            (features['pkd'] - orig_features['pkd'])])
	# Lipinski 조건을 모두 만족하면 보너스
	if lip_pass:
		score += 10
    
	return score

In [22]:
orig_features = predict_feature(smiles, mol)

c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


In [23]:
# make_roulette: 룰렛 휠 선택을 위한 룰렛 생성
# 선택 : 현재 세대의 해들(분자 후보들) 중 어떤 개체를 다음 세대로 전달할지 결정
# 적합도가 높을수록 룰렛에서의 면적이 커짐 (선택될 확률이 높아짐)
def make_roulette(generation):
    fitnesses = [get_fitness(c,Chem.MolFromSmiles(c),orig_features) for c in generation] # 적합도 계산
    
    prev_value = 0.0
    roulette = [0.0]
    for f in fitnesses:
        value = float(f / sum(fitnesses))
        roulette.append(prev_value + value)
        prev_value += value
    
    return roulette

In [24]:
# selection: 룰렛 휠 선택 연산 #
def selection(chronos, roulette):
    selected_chrono = None
    dart = random.random()  # 다트 던지기 EX) 0.27

    # 룰렛에서 해 선택
    for idx in range(1, len(roulette)):
        if dart < roulette[idx]: # dart보다 룰렛 적합도가 높으면
            selected_chrono = chronos[idx-1] # 그 적합도가 높은 chrono를 선택함
            break
    
    return selected_chrono

In [25]:
# crossover: 1점 교차 연산 -> 그냥 부모 2개 랜덤으로 잘라서 이어 붙이는거
def crossover(ca, cb):
	ca2 = list(sf.split_selfies(sf.encoder(ca)))
	cb2 = list(sf.split_selfies(sf.encoder(cb)))
	cross_point = random.randint(1, min(len(ca2),len(cb2))-1)     
	offspring = ca2[:cross_point] + cb2[cross_point:]  
	return offspring

In [26]:
# mutation: 변이 연산 #
def mutation(chrono):
	mutated_chrono = chrono # 이건 
	propability = 0.03      # 변이 확률 0.03
	good_token = ['[O]', '[N]', '[c]', '[OH]', '[=O]']
	branch_list = [i for i in mutated_chrono if 'Branch' in i]

	if random.random() < propability:
		case = random.randint(0,1)
		if branch_list and case == 0:
			mutated_chrono.remove(random.choice(branch_list))
		else:
			mutated_chrono.append(random.choice(good_token))

	# 변이된 문자열 반환
	return sf.decoder(''.join(mutated_chrono))

In [27]:
def sort_generation(generation):
    fitnesses = [get_fitness(c,Chem.MolFromSmiles(c),orig_features) for c in generation]
    sorted_gen = [c for _, c in sorted(zip(fitnesses, generation))]
    return sorted_gen

In [28]:
# sort_generation: 적합도를 기반으로 해집단 정렬
def sort_generation(generation):
    fitnesses = [get_fitness(c,Chem.MolFromSmiles(c),orig_features) for c in generation]
    sorted_gen = [c for _, c in sorted(zip(fitnesses, generation))]
    return sorted_gen # 적합도 낮은 거부터 튀어나옴

# make_offsprings: 부모 세대로부터 자식 세대 생성 #
def make_offsprings(generation):
    ggap = 0.8  # 세대차
    sorted_gen = sort_generation(generation)    # 정렬된 해집단
    n_parents = int(gen_size * (1.0 - ggap))    # 남겨놓을 부모 해의 개수는 3개
    offsprings = sorted_gen[-n_parents:]         # 우수한 부모 해 그대로 남기기 (적합도 높은 상위 3개)
    roulette = make_roulette(sorted_gen)        # 룰렛 생성

    # 남은 수만큼 자식해 생성 후 대치
    for i in range(gen_size - n_parents):
        ca = selection(sorted_gen, roulette)    # 부모해 선택 1 (dart 던져서 적합도가 높은 거1, 근데 dart가 랜덤이라 걍 랜덤으로 추출하는듯)
        cb = selection(sorted_gen, roulette)    # 부모해 선택 2 (dart 던져서 적합도가 높은 거2)
        offspring = crossover(ca, cb)           # 교차
        offspring = mutation(offspring)         # 변이
        offsprings.append(offspring)

    return offsprings

In [29]:
# get_best_chrono: 가장 우수한 해와 그 적합도 반환 #    
def get_best_chrono(chronos):
    fitnesses = [get_fitness(c,Chem.MolFromSmiles(c),orig_features) for c in chronos]
    best_fitness = max(fitnesses)
    best_idx = fitnesses.index(best_fitness)
    return chronos[best_idx], best_fitness

In [ ]:
max_iter = 1
best_fitness = 10
iteration = 0

best_results = []
target = df['dnew_smiles'].iloc[1]

# for target in df['dnew_smiles']:
mol = Chem.MolFromSmiles(target)
generation = [make_random_chrono(target) for _ in range(gen_size)]
best_result = []
# 종료 조건 만족(최적해 발견) 시까지 반복
while best_fitness <= 20 and iteration < max_iter:
	iteration += 1
best_chrono, best_fitness = get_best_chrono(generation)
print('Gen', iteration, '---', 'Best:', best_chrono, 'fitness:', best_fitness)
best_result.append([iteration, best_chrono, best_fitness])
generation = make_offsprings(generation)
best_results.append(best_result)

c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature na

Gen 1 --- Best: FC=CNCCN=CN1CC=NC(Cl)=C1Cl fitness: 13.339999999999998


c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature na

In [31]:
orig_features
# 독성 : 낮을수록 인체에 무해하다는 의미
# qed : 화합물이 약처럼 될 가능성을 수치화(0~1까지)

{'독성 점수': -3.91,
 '분자량': 333.37,
 'logp': 2.43,
 'qed': 0.21,
 'hbd': 0,
 'hba': 7,
 'pki': 7.99,
 'pkd': 9.09}

In [ ]:
best_smiles = best_results[0][0][1]
best_mol = Chem.MolFromSmiles(best_smiles)

predict_feature(best_smiles, best_mol)
# 독성도 낮아지고 약물이 실제 약이 될 확률인 QED도 증가한 것을 통해 부모세대보다 우월하다는 것을 입증

c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


{'독성 점수': -4.18,
 '분자량': 265.12,
 'logp': 2.04,
 'qed': 0.36,
 'hbd': 1,
 'hba': 3,
 'pki': 7.72,
 'pkd': 12.28}